# Practicum 2 (TB142Ic)

Name: Willem Elshout

Dit is de nieuwe Notebook die na de mail van Roel Dobbe is gemaakt, er zal bij bepaalde vragen toegelicht worden hoe het antwoord is verkregen en er zullen kleine vernaderingen bij bepaalde vragen worden toegepast. Dit wordt gedaan omdat er bij sommige vragen AI is gebruikt als tool, wel snap ik dat het belangrijk is om de code ook te begrijpen zodat er een leerproces is, daarom is er bijgezet wat er gebeurd met de code en hoe deze verkregen is.

## Objective of this assignment

The objective is to apply concepts about statistical tests and bootstrapping, and to get familiar with performing machine learning using Scikitlearn. Upon completing this exercise it is expected that you:

- Are well versed at computing confidence intervals and doing statistical tests, and using the bootstrap method.
- Have an understanding of how to implement linear regression.
- Have an appreciation of feature engineering - adding features to your model to improve the predictive capability.
- Understand the concept of under- and overfitting.

## Advised Reading and Exercise Material

- See "Stof Week 2" in the Content module for Week 2 on Brightspace.

## ** Important: ** When handing in your homework:
 
+ Hand-in: upload to Brightspace.
+ Notebook must be handed in indivudally
+ Use the practicum sessions and tutorial sessions on Wednesday and Friday, and Brightspace discussion board for questions on how to complete the exercises.
+ If you find mistakes/have suggestions/would like to discuss the assigment material itself, please mail [Maarten] at `[course email]`.

## 0. Importeren van packages

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn import linear_model
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import scipy.stats as sps

## 1. Modelselectie (32 punten)
Hieronder is een dataset gegeven van $15$ $x$- en $y$-waarden. De datapunten bevinden zich tussen $x=-5$ en $x=5$ en bevatten een (kleine) meetfout door ruis. De dataset is weergegeven in de plot hieronder met zwarte bolletjes. Daarnaast zijn er drie regressielijnen weergegeven. Het gaat hier om regressies aan polynomen, dus aan functies $y=a_0 + a_1x + a_2x^2+\cdots+a_nx^n$, waarbij $n$ de orde wordt genoemd. Hoe hoger de orde, hoe hoger het aantal vrijheidsgraden van de functie (ook wel 'flexibiliteit' of 'degrees of freedom' in het Engels).

De variabele `x_fit` bevat $200$ $x$-waarden tussen $-5$ en $5$ om een plot mee te maken. De variabele `y_fit` bevat een lijst met numpy arrays. Iedere array bevat de $y$-waarden corresponderend met een regressielijn. De $y$-waarden van de vierde orde polynoom krijg je zo: `y_fit[4]`. Dit is gedaan tot orde $14$.

De variabele `y_pred` is een lijst met de voorspelde $y$-waarden voor iedere orde polynoom. Het verschil tussen `y_fit` en `y_pred` is de $x$-waarde waarvoor de voorspelling is gedaan. `y_pred` kun je bijvoorbeeld gebruiken om de errors te berekenen.

In [ ]:
# Deze cell kun je niet aanpassen, maar moet je wel een keer runnen!

x = np.array([ 1.89564137,  2.1950297 , -2.43068177,  4.24280261, -0.57083932,
              4.72729678,  0.00999159, -4.06773877,  2.16876064, -2.99045758,
              -2.82309756,  1.03261698,  4.13025458, -0.63768357, -4.99470922])
y = np.array([ 0.96968846,  1.4014468 , -1.07360771,  1.64834086, -0.67485827,
              1.61423619, -0.22877331, -1.20824454,  0.90316278, -0.96462048,
              -1.00772175,  0.5721553 ,  1.12609029, -0.57714603, -1.29119042])

x_fit = np.linspace(-5,5,200)
y_fit = [np.polyval(np.polyfit(x,y,n), x_fit) for n in range(15)]

y_pred = [np.polyval(np.polyfit(x,y,n), x) for n in range(15)]

fig_data, ax_data = plt.subplots()

ax_data.scatter(x, y, c='k')
ax_data.plot(x_fit,y_fit[1],x_fit,y_fit[4],x_fit,y_fit[12])
ax_data.set_ylim(-2,2); ax_data.set_xlim(-5,5)
ax_data.legend(['Data','1e orde','4e orde','12e orde'])
ax_data.set_xlabel('$x$')
ax_data.set_ylabel('$y$')
fig_data;

---
**1.1.** Hoeveel vrijheidsgraden heeft de twaalfde orde polynoom? 
Omdat je maar 15 datapunten hebt is de orde van de polynoom beperkt. Wat is de hoogste orde polynoom die je met 15 datapunten zou kunnen maken? Leg uit. **(6pt)**

**Solution**: 
De vrijheidsgraden van een polynoom geven het aantal onafhankelijke variabelen weer dat kan worden aangepast bij het bepalen van de polynoomcoëfficiënten. Voor een polynoom van de twaalfde orde zijn er 13 coëfficiënten die kunnen worden aangepast (één voor elke term in de polynoom).

Met slechts 15 datapunten is het belangrijk om voorzichtig te zijn bij het kiezen van de orde van de polynoom om overfitting te voorkomen. Overfitting treedt op wanneer een polynoom te complex is voor het aantal beschikbare datapunten, waardoor de polynoom te goed past bij de gegevenspunten maar niet generaliseert naar nieuwe gegevens.

Er is geen vaste regel voor het bepalen van de hoogste orde polynoom die je met 15 datapunten kunt maken. Het is echter gebruikelijk om de regel van duim te volgen dat de orde van de polynoom niet hoger mag zijn dan het aantal datapunten minus één. In dit geval zou de hoogste orde polynoom die je met 15 datapunten zou kunnen maken een polynoom van de veertiende orde zijn (15 - 1). Het is echter belangrijk om te onthouden dat het gebruik van een polynoom van deze orde mogelijk niet de beste benadering is, omdat het risico op overfitting groot is. In dergelijke gevallen worden vaak alternatieve benaderingen zoals lineaire regressie of het gebruik van andere modellen aanbevolen om een betere balans te vinden tussen modelcomplexiteit en beschikbare gegevens

---
**1.2.** Wat betekent het als een model 'overfit' is? Gebruik dit feit om uit te leggen welke van deze drie regressielijnen in de bovenstaande plot jouw voorkeur heeft. **(4pt)**

**Solution**: Wanneer een model te complex is voor de gegevens die het probeert te modelleren, kan het fenomeen optreden dat we 'overfitting' noemen. Overfitting treedt op wanneer een model te veel van de ruis en variabiliteit in de trainingsgegevens weerspiegelt, in plaats van het algemene patroon van de gegevens vast te leggen. Als gevolg hiervan past het model heel goed bij de trainingsgegevens, maar heeft het de neiging om slecht te presteren op nieuwe gegevens die niet in de trainingsset zijn opgenomen.

Een model dat overfit is, heeft zich te sterk aangepast aan de trainingsgegevens en heeft daardoor onvoldoende rekening gehouden met de onderliggende relatie tussen de variabelen. Het model kan mogelijk complexe interacties tussen variabelen hebben aangeleerd die niet aanwezig zijn in de werkelijke populatie. Dit leidt vaak tot een slechte algemene prestatie van het model op nieuwe, ongeziene gegevens.

Overfitting kan worden voorkomen door het model te vereenvoudigen, door bijvoorbeeld het aantal parameters te verminderen of door regularisatie toe te passen. Het is belangrijk om op te merken dat hoewel het verminderen van overfitting ten koste kan gaan van de prestaties op de trainingsgegevens, het doel van een model is om nieuwe, ongeziene gegevens zo nauwkeurig mogelijk te voorspellen, niet alleen de trainingsgegevens.

In dit geval lijkt een polynoom van de 4e orde een goede keuze te zijn, omdat het een compromis biedt tussen het lineaire model en de 12e orde polynoom. Het heeft voldoende complexiteit om de onderliggende structuur van de gegevens beter vast te leggen dan een eenvoudig lineair model, maar het is niet zo complex dat het gevoelig is voor overfitting, zoals een polynoom van de 12e orde. Daarom zou ik de voorkeur geven aan de 4e orde polynoom in deze situatie, omdat het een evenwichtige benadering biedt tussen modelcomplexiteit en het beperkte aantal datapunten dat beschikbaar is.

---
**1.3.** Maak een plot van de root mean square error (RMSE) tegen de orde van de regressielijn. Geef een label aan je assen en verzorg je plot van een titel. De RMSE bereken je als volgt: $\text{RMSE}=\sqrt{\frac{1}{n}\sum_{i=0}^{n-1}(y-y_\text{pred})^2}$. **(4pt)**

In [ ]:

from sklearn.metrics import mean_squared_error
#de mean_squared_error wordt geimporteerd van de skylearn libary
RMSE = []
for n in range(15):
    y_pred = np.polyval(np.polyfit(x,y,n), x)
    RMSE.append(np.sqrt(mean_squared_error(y, y_pred)))
#de orde van de regressielijn wordt weergegeven en gaat tot de 15 omdat er geen hogere ordes meer zijn, de RMSE wordt steeds berekend van een orde, de orde wordt steds 1 groter

fig_RMSE, ax_RMSE = plt.subplots()
ax_RMSE.plot(range(15), RMSE)
ax_RMSE.set_xlabel('Orde van de regressielijn')
ax_RMSE.set_ylabel('RMSE')
ax_RMSE.set_title('RMSE voor verschillende orden van de regressielijn')
fig_RMSE;
#vervolgens wordt er een plot gemaakt met range 15 voor de orde, de labels van de assen worden toegevoegd en de titel wordt toegevoegd.


---
Hieronder zijn nog eens $15$ meetwaarde gegeven. Deze zullen worden gebruikt om de regressielijnen mee te testen. Deze datapunten zijn opgeslagen in `x_t` en `y_t`. De voorspelde $y$-waarde voor iedere waarde in `x_t` is opgeslagen in arrays in `y_t_pred`. Net als `y_pred` is deze variabele een lijst van arrays waar de lijstindex de orde van de polynome regressie aangeeft. Eenzelfde plot als hierboven is weergegeven, maar nu zijn ook de testdata weergegeven.

In [ ]:
# Deze cell kun je niet aanpassen, maar moet je wel een keer runnen!
x_t = np.array([ 1.0937798 ,  0.74369275, -3.65096628, -1.77817436,  1.21922561,
       -2.37943093, -0.57636558,  4.08939144,  1.96429294, -0.80976745,
       -4.14756781, -2.4385666 ,  2.72470686,  3.73580783, -3.22004025])
y_t = np.array([ 0.90598778,  0.28059777, -1.51761398, -1.38347669,  0.96414018,
       -0.90840507, -0.28085167,  1.69742952,  1.02036716, -0.91035683,
       -1.65803719, -0.95699662,  1.3602569 ,  1.3648236 , -1.01638113])
y_t_pred = [np.polyval(np.polyfit(x,y,n), x_t) for n in range(15)]


ax_data.scatter(x_t, y_t, c='r',label='Test data')
ax_data.legend(['Trainingsdata','1e orde','4e orde','12e orde','Testdata'])
fig_data

In [ ]:
np.polyfit(x,y,5)

---
**1.4.** Bereken voor iedere orde nu ook de RMSE van de testdata. Maak vervolgens ook een correcte plot van de RMSE van de trainingsdata én de testdata. Beperk de $y$-as tot het interval [0,2]. **(5pt)**

In [ ]:
RMSE_test = np.sqrt(np.mean((y_t - y_t_pred)**2))
RMSE_train = []
RMSE_test = []
#er wordt een lijst aangemaakt van train data en test data
for n in range(15):
    y_pred_train = np.polyval(np.polyfit(x, y, n), x)
    y_pred_test = np.polyval(np.polyfit(x, y, n), x_t)
    RMSE_train.append(np.sqrt(np.mean((y - y_pred_train)**2)))
    RMSE_test.append(np.sqrt(np.mean((y_t - y_pred_test)**2)))
#de orde van ed regressielijn wordt weergegeven tot de 15 
fig_rmse, ax_rmse = plt.subplots()
ax_rmse.plot(range(15), RMSE_train, 'b', label='Trainingsdata')
ax_rmse.plot(range(15), RMSE_test, 'r', label='Testdata')
ax_rmse.set_ylim(0, 2)
ax_rmse.legend()
ax_rmse.set_xlabel('Orde van de regressielijn')
ax_rmse.set_ylabel('RMSE')
ax_rmse.set_title('RMSE van de trainings- en testdata voor verschillende ordes van de regressielijn')
#er wordt een nieuwe plot gemaakt met niuwe labels in de plot met een legande per plot
fig_rmse

---
**1.5.** Beschrijf wat je in bovenstaande grafiek ziet. Zie je een verschil tussen de twee lijnen? Waar komt dit door? **(6pt)**

**Solution**: Het lijkt erop dat de trainingsdata-RMSE begint met een redelijk hoge waarde, daalt tot een minimum en begint vervolgens weer te stijgen, zelfs tot oneindig hoog bij hogere ordes. Dit is een klassiek teken van overfitting. Wat er gebeurt, is dat naarmate het model complexer wordt (met een hogere orde polynoom), het model begint met het 'leren' van de ruis in de trainingsdata, wat resulteert in een lagere fout op de trainingsdata maar een hogere fout wanneer we het model testen op nieuwe gegevens.

Aan de andere kant, de testdata-RMSE begint ook bij een hoge waarde, daalt naar een minimum en blijft daarna verder afnemen naar 0. Dit geeft aan dat het model steeds beter past bij de testdata naarmate de orde toeneemt. Dit is enigszins ongewoon, omdat we normaal gesproken zouden verwachten dat de test-RMSE zou stijgen bij hogere ordes vanwege overfitting, zoals we zien bij de trainingsdata.

Het verschil tussen de twee lijnen kan mogelijk worden toegeschreven aan het verschil in kenmerken tussen de trainings- en testdata. Het is mogelijk dat de trainingsdata meer ruis bevatten of complexer zijn dan de testdata. Het is ook mogelijk dat de testdata toevallig goed passen bij de hogere orde polynomen, wat niet noodzakelijkerwijs het geval zou zijn met andere testdatasets.

---
**1.6.** Welk model heeft de kleinste RMSE voor de testdata en wat is deze waarde? Gebruik hiervoor de onderstaande cell: **(4pt)**

In [ ]:
best_model = np.argmin(RMSE_test) + 1
best_RMSE = RMSE_test[best_model - 1]
#het model met de kleinste RMSE waarde wordt benoemd en de kleinste waarde van de RMSE wordt als laatste uit een lijst gepakt
print(f"Het model met de kleinste RMSE voor de testdata is het {best_model}e orde model, met een RMSE van {best_RMSE:.2f}")

---
**1.7.** De meetingen bevatten normaalverdeelde ruis met gemiddelde $0$ en standaarddeviatie $\sigma=0.25$. Hoe verhoudt de kleinste RMSE zich tot de ruis? Geeft dit model een adequate beschrijving van de data of zou je een ander (niet-polynoom) model willen onderzoeken? Met andere woorden: is er veel verbetering mogelijk? (LET OP: je hoeft hiervoor niet verder het model te bestuderen, enkel de RMSE waarde te vergelijken met de standaarddeviatie van de ruis) **(3pt)**

**Solution**: De laagste RMSE-waarde voor de testdata is 0.26, terwijl de standaarddeviatie van de ruis 0.25 is. Dit betekent dat het model erin slaagt om de variabiliteit veroorzaakt door de ruis in de metingen te overtreffen en een redelijk goede beschrijving van de data te geven. Het model lijkt de gegevens goed te passen en heeft een lage fout bij het voorspellen van de testdata.

Niettemin is het altijd verstandig om andere modellen te verkennen om te zien of er nog ruimte is voor verbetering. Het kan zijn dat een ander model, dat geen polynomiale vorm heeft, mogelijk beter aansluit bij de aard van de gegevens en een nog nauwkeurigere voorspelling kan bieden. Het is belangrijk om open te staan voor alternatieve modellen en deze te onderzoeken om ervoor te zorgen dat we het best mogelijke model hebben voor onze gegevens.


---
## 2. Regressie-analyse (50pt)
Steeds meer weersvoorspellingen maken gebruik van machine learningtechnieken. Dit gebeurt voor voorspellingen op zowel de korte als de lange termijn. In deze opdracht maken we een zeer lange termijnvoorspelling van de aantal uren zonlicht in De Bilt.
### 2.1 Data verkennen (10pt)
**2.1.1.** Laad de dataset van het KNMI met de uren zonlicht 'KNMI_20221231.txt' in. De eerste kolom bevat de code voor het meetstation. Omdat deze dataset enkel metingen bevat van het meetstation in De Bilt (code 260), kun je deze kolom negeren. 

> **Tip**: Bekijk de documentatie van de `read_csv`-functie voor hoe je slechts bepaalde kolommen inlaadt. `read_csv` werkt ook voor `.txt`-files.

Gebruik een parser om de datum van iedere meting in te lezen via de twee functie inputs `parse_dates` en `date_parser`. Voor `dateparser` kun je een `lambda` functie gebruiken (ken je die?). 
Zorg ervoor de datumkolom ook de index is van het dataframe, oftewel de meest linkse kolom bevat de datum (dit doe je met de input `index_col=...`). 

Tot slot, de data is gemeten in 0.1 uren, dus deel de waarden door 10 om de tijd in uren te krijgen. **(4pt)**
>**Tip**: Om de documentatie van een functie te bekijken, type je deze eerst uit, ga je op de functie staan met de cursos en druk je Shift+Tab. Druk vervolgens op het plusje om de documentatie goed te lezen. Dit is een goede gewoonte om beter begrip te vormen van alle functies waar je code van afhankelijk is en een belangrijke manier om fouten uit je code te halen / te debuggen.

In [ ]:
date_parser = lambda x: pd.to_datetime(x.strip(), format='%Y%m%d')
#een lambda functie is gemaakt om de datum te parsen
df = pd.read_csv('KNMI_20221231.txt', 
                 sep=',', 
                 index_col='YYYYMMDD', 
                 usecols=['YYYYMMDD','SQ'], 
                 parse_dates=['YYYYMMDD'], 
                 date_parser=date_parser,
                 skipinitialspace=True)  
#de dataset wordt geladen en de datum wordt ingelezen door middel van de funcites parsedates en dateparser
df['SQ'] = df['SQ'] / 10
#de dataset is gemeten in 0.1 uren dus de waarden moeten worden gedeeld worden door 10 om het in uren te krijgen.

---
**2.1.2.** Maak een volledige plot van de dataset. Als het goed is heb je jaartallen op de x-as staan. **(4pt)**

In [ ]:
ax = df.plot(figsize=(10,5))
# er wordt een plot gemaakt van de dataset
ax.set_xlabel('Jaar')
ax.set_ylabel('SQ (mm)')
#er worden labels aan de assen gegeven
plt.title('Neerslaghoeveelheid in De Bilt (1901-2022)')
#er wordt een titel gegeven aan de plot

---
**2.1.3.**
Welke factoren denk je dat de vorm van deze data verklaart? Noem tenminste twee factoren. **(2pt)**

**Solution:** De vorm van de neerslagdata in De Bilt van 1901 tot 2022 kan door verschillende dingen worden verklaard. Twee van die dingen zijn het seizoen en klimaatverandering. Het seizoen heeft invloed op de neerslag. In Nederland hebben we veel regen als er lagedrukgebieden zijn, vooral in de herfst en winter. Dus het is niet gek als de wintermaanden in De Bilt natter zijn dan de zomermaanden. Maar ook klimaatverandering speelt een rol. Veranderingen in het wereldwijde klimaat kunnen ervoor zorgen dat het overal op de wereld meer of minder regent. Dat hangt af van dingen zoals de luchtstromen en de temperatuur van de oceanen. Dus het kan zijn dat de neerslag in De Bilt de afgelopen jaren is veranderd door klimaatverandering.Dus als we kijken naar de neerslagdata, moeten we rekening houden met het seizoen en klimaatverandering. Die kunnen allebei een rol spelen in hoeveel het regent in De Bilt over een lange periode.

---
### 2.2 Bouwen van regressie-analyse (40pt)
Voor de regressie-analyse maken we gebruik van de package scikit-learn (`sklearn`). We zullen gebruik maken van een lineaire regressie. Dat betekent dat we data proberen te 'fitten' aan een functie zoals $y = a_0+a_1x_1+a_2x_2+a_3x_3$. Hier gebruiken we de 'predictors' $x_1$, $x_2$ en $x_3$ en de output $y$. De predictors kunnen allerlei numerieke waarden zijn zoals de tijd, de hoeveelheid regen en het aantal verkochte ijsjes. Voor de parameters $a_1$, $a_2$ en $a_3$ proberen we met machine learning de waarden te vinden.

Als we bijvoorbeeld de functie $y = a_0 + a_1x + a_2x^2$ will fitten, hebben we de predictors $x_1 = x$ en $x_2=x^2$. We zien $x^2$ eigenlijk als een nieuwe variabele.

**2.2.1.** Eerst proberen we de data te verklaren aan de hand van de eenvoudige functie $y = ax + b$, waar $y$ de maximum temperatuur is en $x$ de dag sinds 1 januari 1990. Hoe kun je $a$ en $b$ interpreteren in deze formule? **(2pt)**

**Solution**: De a is het hellingsgetal en de b het begingetal

---
**2.2.2.** Reken eerst de datum in de datumkolom om naar het aantal dagen sinds 1 januari 1990. **(2pt)**
>**Tip**: Je kunt verschillen uitrekenen tussen datetimes door ze eenvoudig van elkaar af te trekken en om te rekenen in dagen: `(datetime(2020,11,24)-datetime(1990,1,1)).days`. De datum van iedere meting is (als het goed is) opgeslagen als index en kun je vinden met `.index`.

In [ ]:
start_datum = datetime(1990, 1, 1)
#dit is de startdatum
df['dagen_sinds_start'] = (df.index - start_datum).days
#er wordt ebrekend heoveel dagen ervorbij zijn gegaan sinds de startdatum

---
**2.2.3.**
We slaan de predictors op in een variabele `X`. Dit is een 2D-array (ofwel een matrix) met alle waardes voor $x_1, x_2$, etc. als kolommen. Voor het geval $y = ax + b$, waarbij $x$ de dagen sinds 1 januari 1990 voorstelt, hebben we één variabele en dus één kolom in `X`. **(2pt)**
>**Tip**: Je kunt in Numpy lege arrays aanmaken met `np.zeros((n,m))`, waarbij $n$ het aantal rijen is en $m$ het aantal kolommen. Maak eerst een 2D-array aan met 1 kolom.

>**Tip**: Indexeren in 2D-arrays gaat op vergelijkbare wijze als met 1D-arrays: `X[i,j]` geeft het element van de $i^e$ rij en $j^e$ kolom. Daarnaast is `X[i,:]` de gehele $i^e$ rij. Op een vergelijkbare manier kun je een kolom indexeren.

In [ ]:
X = np.zeros((len(df), 1))
#er wordt een array aangemaakt met 1 kolom
X[:, 0] = df['dagen_sinds_start']
#er wordt geindexeerd, 0de kolom

In [ ]:
display(X)
X.shape

---
In onderstaande cell wordt de gehele dataset gesplitst in test- en trainingsdata. De metingen van 2016 en later worden gebruikt om het model te testen.

In [ ]:
sep = temp.index.get_loc(datetime(2016,1,1))
X_train = X[:sep, :]
y_train = temp.to_numpy()[:sep]
X_test  = X[sep:, :]
y_test  = temp.to_numpy()[sep:]

Het is nu tijd om het model te maken en te 'fitten'. Eerst maken we een leeg model met `linear_model.LinearRegression()`. Dit object heeft een aantal functies die belangrijk zijn in het machine learning proces. Ten eerste wordt `.fit(X,y)` gebruikt om het model te trainen ('fitten'). De parameterwaarden zijn nu gevonden en zijn op te vragen met `.coef_`. De intercept is opgeslagen in `.intercept_`. Tot slot kun je voorspellingen doen met je model met `.predict(X)`. De documentatie is terug te vinden op https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression.

---
**2.2.4.** Maak nu een 'leeg' model aan en train het met `.fit(..., ...)`, waarbij je de trainingdata gebruikt. **(3pt)**

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
#er wordt een nieuw model aangemaakt en de traiingsdata wordt er ingeladen

---
**2.2.5.** Print de parameters die je hebt gevonden (zowel de coëfficienten als de intercept).**(4pt)**

In [ ]:

print("Coëfficiënten:")
for i, coef in enumerate(model.coef_):
    print(f"𝑎{i+1}: {coef}")
    #de coeficcienten worden opgevraagd
print("Intercept:")
print(f"𝑏: {model.intercept_}")
#de intercept wordt opgevraagd in het model

---
**2.2.6.** Is dit denk je een goed model voor onze data? Leg uit. **(2pt)**

**Solution:**Op dit moment hebben we het model alleen getraind op de beschikbare gegevens, maar we hebben het nog niet getest op nieuwe gegevens. Het is pas echt goed als het model in staat is om accuraat te voorspellen op gegevens die het nog nooit eerder heeft gezien. We kunnen de voorspelde waarden vergelijken met de werkelijke waarden om te zien hoe goed het model presteert. Daarnaast zijn er andere statistieken, zoals de R-kwadraatwaarde (R^2) en de gemiddelde absolute fout (MAE), die ons kunnen helpen beoordelen hoe goed het model het doet met nieuwe gegevens.

---
**2.2.7.** Maak nu een plot met de werkelijke metingen vanaf 2016 en de voorspelling (Tip: gebruik `temp.index[index_1jan2016:].to_numpy()` voor de tijdsreeks, waarbij `index_1jan2016` al hierboven is gedefinieerd). Zorg ervoor dat de figuur alle elementen bevat en dat de $x$-as de datum weergeeft (i.p.v. het aantal dagen sinds 1990). 
Hoe goed doet het model het? Wat heeft het wel en wat heeft het niet goed voorspelt? **(7pt)**

In [ ]:
y_pred = LinearRegression.predict(, X)
#de voorspellingen
dates = temp.index[sep:].to_numpy()
#de date
plt.plot(dates, y_test, label='Werkelijke metingen')
#dit zijn de echte metingen
plt.plot(dates, y_pred, label='Voorspelling')
#hierboven worden de voorspellingen geplot
plt.xlabel('Datum')
plt.ylabel('Temperatuur')
plt.title('Werkelijke metingen vs. Voorspelling')
#hierboven worden xas en yas labels toegevoegd en de titel
plt.legend()
#er wordt een legenda toegevoegd
plt.show()


**Solution:**

---
We gaan nu de vorige stappen herhalen voor twee nieuwe 'features'. Het is duidelijk dat er een cyclisch patroon in de data te vinden is, welke we niet met het simpele model uit de data kunnen 'fitten', oftwel ons model is niet flexibel genoeg. We gaan het cyclische patroon proberen op te nemen in ons model. Daartoe voegen we een sinus- en een cosinusterm toe aan de functie. 

**2.2.8.** Een sinusoide $a\cdot\sin(b\cdot x)$ heeft periode $\frac{2\pi}{b}$. Wat is $b$ (exact) voor een periode van 365.25 dagen (=1 jaar)? **(1pt)**

**Solution:** Als we kijken naar een tijdsperiode van 365.25 dagen, dan zien we dat de waarde van 𝑏 ongeveer 0.0172022464 is (afgerond op 10 decimalen). Dit betekent dat elke extra dag in die periode een heel klein effect heeft van ongeveer 0.0172022464 op de voorspelde waarde. Met andere woorden, voor elke dag die we toevoegen aan de periode, voegt het model een kleine waarde toe aan de voorspelling.

---
De uiteindelijke functie ziet er nu als volgt uit: $y=a_0+a_1\cdot d+a_2\cdot\sin(\ldots d)+a_3\cdot \cos(\ldots d)$. Hier is $d$ het aantal dagen sinds 1 januari 1990. 

**2.2.9.** Maak een nieuwe variabele aan zoals `X`, maar nu met 3 kolommen. Plaats in de eerste kolom het aantal dagen sinds 1990. Sla het resultaat voor de twee nieuwe features op in de tweede en derde kolom ($x_2 = \sin(\ldots d)$ en $x_3 = \cos(\ldots d)$). **(4pt)**
>**Tip**: Je kunt de sinus- en cosinusfuncties van numpy gebruiken: `np.sin(...)`. Verder is ook $\pi$ beschikbaar door numpy: `np.pi`.

In [ ]:
X = np.zeros((len(df), 3))
#er is een nieuwe array met 3 kolommen 
X[:, 0] = df['dagen_sinds_start']
#deze kolom geeft aan hoeveel dagen er nog zijn sinds 1 jan 1990
X[:, 1] = np.sin(2 * np.pi * df['dagen_sinds_start'] / 365.25)
#Deze kolom geeft de sinus aan van het nog aantal dagen tot 1 jan 1990.
X[:, 2] = np.cos(2 * np.pi * df['dagen_sinds_start'] / 365.25)
#Deze kolom geeft de cosinus aan van het aantal dagen tot 1 jan 1990.

---
**2.2.10.** Verdeel nu ook de data in test- en trainingsdata zoals hierboven. **(2pt)**
>**Tip**: gebruik de variable `sep` opnieuw om de data te splitsen in tot en vanaf 1 januari 2016. Je kunt ook de `y` variabelen hergebruiken.

In [ ]:
X_train = X[:sep, :]
y_train = df['SQ'].to_numpy()[:sep]
#de data wordt verdeeld hierboven zie je de trainingsdata
X_test = X[sep:, :]
y_test = df['SQ'].to_numpy()[sep:]
#de data wordt evrdeeld hierboven zie je de test data

---
**2.2.11.** Maak nu een nieuw 'leeg' model aan voor de drie features en train het model, waarbij je de trainingdata gebruikt. **(3pt)**

In [ ]:
model_new = LinearRegression()
model_new.fit(X_train, y_train)
#er wordt een nieuw model aangemaakt en de trainingsdata wordt erin gezet

---
**2.2.12.** Print de parameters die je hebt gevonden (zowel de coëfficienten als de intercept). Wat valt je op aan wat het model met de eerste feature $d$ doet? Waarom is dit denk je? **(3pt)**

In [ ]:
print("Coefficients: ", model.coef_)
print("Intercept: ", model.intercept_)
# de coefficienten en de parameter worden geprint

**Solution:** Als we de parameters van het getrainde model willen afdrukken, kunnen we de attributen "coef_" en "intercept_" van het regressie-object gebruiken. Wat opvalt, is dat de coëfficiënt voor de eerste feature (d) aanzienlijk kleiner is dan die voor de sinus- en cosinus-termen. Dit suggereert dat het model minder nadruk heeft gelegd op de lineaire trend in de data. Waarom? Omdat de cyclische patronen in de data veel sterker zijn dan de lineaire trend. Het model heeft zich meer gericht op het modelleren van deze cyclische patronen door gebruik te maken van de sinus- en cosinus-termen, waardoor de lineaire term minder belangrijk is geworden.

**2.2.13.** Maak nu een plot met werkelijke metingen vanaf 2016 en de voorspelling zoals in 2.2.7. **(5pt)**

In [ ]:
df = pd.read_csv('KNMI_20221231.txt', sep=',', parse_dates=['YYYYMMDD'], index_col='YYYYMMDD')
df['   SQ'] = df['   SQ'] / 10  # Converteer de gemeten uren zonlicht van 0.1 uur naar uur
#de dataset wordt ingeladen
d = (df.index - datetime(1990, 1, 1)).days
X = np.zeros((len(df), 3))
X[:, 0] = d
X[:, 1] = np.sin(2 * np.pi * d / 365.25)
X[:, 2] = np.cos(2 * np.pi * d / 365.25)
#er wordt een dataset aangemaakt met 3 verschillende klommen
sep = df.index.get_loc(datetime(2016, 1, 1))
X_train = X[:sep, :]
y_train = df['   SQ'].to_numpy()[:sep]
X_test = X[sep:, :]
y_test = df['   SQ'].to_numpy()[sep:]
#de datase wordt opgesplitst in test en trainingsdata
model_new = LinearRegression()
#er wordt een nieuw model aangemaakt
model_new.fit(X_train, y_train)
#de trainingsdata wordt erin gezet
y_pred = model_new.predict(X_test)
#de data voor de testdata wordt voorspeld
plt.plot(df.index[sep:], y_test, label='Werkelijke metingen')
plt.plot(df.index[sep:], y_pred, label='Voorspelling')
plt.xlabel('Datum')
plt.ylabel('Aantal uren zonlicht')
plt.legend()
plt.title('Werkelijke metingen vs. Voorspelling')
plt.xticks(rotation=45)
plt.show()

#hierboven worden de werkelijke metingen geplot die nu hieronder in de grafiek te zien zijn, er is een label voor de x as yas er is een titel

---
### 3. Bonus (14pt)
**3.1a** Hoe lang schijnt de zon op de dag met de meeste uren zon volgens jouw voorspelling in de periode 2016 t/m 2022? Geef je antwoord in uren afgerond op 2 decimalen.**(2pt)** 
> Tip: Zoek `string formatting python` op en gebruik dit om je antwoord te printen: "Volgens mijn voorspelling is..."

**3.1b** Interpreteer jouw voorspelling afgezet tegen de waarden in de data. Kijk nog eens goed naar de daadwerkelijke waarde. Durf je te leunen op jouw voorspelling als het een advies zou zijn, bijvoorbeeld om het weer te voorspellen? Waarom wel/niet? **(2pt)**

**Solution**:

---
**3.2** Plot de residual error signal (trek je voorspelling af van de data) met een histogram met 50 bins. Is het acceptabel aan te nemen dat de residual bij benadering normaal gedistribueerd is? Plot vervolgens ook de residual over de tijd heen en leg uit of dit jouw antwoord bevestigd of niet. **(5pt)**

**Solution**:

**3.3** Plot de residual error signal van alleen het jaar 2021 (1 januari t/m 31 december). Welke patronen herken je hierin? Leg uit wat deze patronen kan veroorzaken (gebruik je inhoudelijke achtergrondkennis over dit onderwerp!). **(5pt)**
> **Tip**: In de array met errors uit de vorige opdracht correspondeert index `0` met 1 januari 2016.

**Solution:** 